# Part 1: Dataset Inspection & Preprocessing Pipeline
**Multi-Layer Phishing Detection & Risk Assessment**

This notebook walks through the data loading, data validation, label mapping verification, and end-to-end cleaning pipeline.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

# Add project root to sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_loader import load_dataset, inspect_dataset
from src.preprocessing import clean_dataset, save_cleaned_dataset

## 1. Load and Inspect Raw Dataset
We load `phishing_url_dataset.csv` and inspect its shape, columns, missing values, duplicates, and initial label frequencies.

In [ ]:
# Resolve raw dataset path
data_path = project_root / "dataset" / "raw" / "phishing_url_dataset.csv"
if not data_path.exists():
    data_path = project_root / "phishing_url_dataset.csv"

raw_df = load_dataset(data_path, standardize_col_names=False)
summary = inspect_dataset(raw_df, verbose=True)

## 2. Verify Label Encoding
In the raw dataset:
- `Label = 0` contains phishing URLs (e.g., dynamic DNS, credential harvesting, fake banking links).
- `Label = 1` contains legitimate URLs (e.g., government, academic, and registered commercial websites).

Our standardized specification requires:
- `0` = Legitimate
- `1` = Phishing

Let's inspect sample records from each raw class to verify this ground truth mapping.

In [ ]:
print("Raw Label = 0 Samples (Phishing):")
print(raw_df[raw_df['Label'] == 0]['URL'].head(5).values)

print("\nRaw Label = 1 Samples (Legitimate):")
print(raw_df[raw_df['Label'] == 1]['URL'].head(5).values)

## 3. Run Reusable Data Preprocessing Pipeline
The pipeline performs:
1. Removal of null / non-string URLs.
2. Trimming surrounding whitespace.
3. Removal of malformed URLs.
4. Standardization of labels to binary format (`0` = Legitimate, `1` = Phishing).
5. Removal of exact duplicate records.
6. Explicit handling and removal of contradictory / conflicting URLs.
7. Deduplication of repeated URLs.
8. Final validation of zero remaining null or invalid values.

In [ ]:
cleaned_df, stats = clean_dataset(raw_df, verbose=True)

## 4. Save Cleaned Dataset

In [ ]:
out_path = project_root / "dataset" / "processed" / "cleaned_dataset.csv"
save_cleaned_dataset(cleaned_df, out_path)
print(f"Cleaned dataset shape: {cleaned_df.shape}")
print(cleaned_df.head(10))